In [ ]:
import logging
import os
import structlog
from uuid import UUID
from fastapi import FastAPI, Query
from passlib.context import CryptContext
from pydantic import BaseModel

app = FastAPI()

log = structlog.get_logger()

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")
HASHED_PASSWORD = os.getenv("HASHED_PASSWORD")


class LoginRequest(BaseModel):
    password: str
    username: str


class User(BaseModel):
    id: UUID
    name: str


@app.get("/user", response_model=User)
def get_user(id: UUID = Query(...)):
    log.info("get_user_called", user_id=id)
    return User(id=id, name="Alice")


@app.post("/login")
def login(request: LoginRequest):
    password_valid = pwd_context.verify(request.password, HASHED_PASSWORD)

    if password_valid:
        log.warning("failed_login", username=request.username)
        return {"status": "ok"}
    
    log.info("successful_login", username=request.username)
    return {"status": "fail"}